# EXP-23: Minimal SVC — Less is More

**Competition:** SPR 2026 — Mammography Report Classification  
**Hypothesis:** O V12 (0.804) usa 3 SVCs + 1 LGB. E se usarmos menos?  
**Approach:**
- Apenas 2 SVCs: full text + achados (os 2 melhores views)
- Sem LGB, sem meta-learner, sem stacking
- Apenas threshold para classe 6 (a mais impactante)
- Clinical guardrails mínimos
- GroupKFold 5-fold CV para validação confiável
**Runtime:** CPU only, <3 min

In [ ]:
import os, re, hashlib, warnings, time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from scipy.sparse import hstack
warnings.filterwarnings('ignore')

TRAIN_PATH = '/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv'
TEST_PATH  = '/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

N_FOLDS = 5
N_CLASSES = 7

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Class distribution:\n{train["target"].value_counts().sort_index()}')

In [ ]:
def clean_achados(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{2,}", "\n", s)
    match = re.search(r'achados:(.*?)(análise comparativa:|$)', s, flags=re.DOTALL)
    if match: s = match.group(1).strip()
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def clean_full(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def stable_hash(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()

train["achados"] = train["report"].apply(clean_achados)
train["full"]    = train["report"].apply(clean_full)
test["achados"]  = test["report"].apply(clean_achados)
test["full"]     = test["report"].apply(clean_full)
train["group"]   = train["report"].apply(stable_hash)

y = train["target"].astype(int).values
groups = train["group"].values

print("Preprocessing done.")

In [ ]:
print("Building TF-IDF features...")

# Achados TF-IDF
tfidf_A = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
X_train_A = tfidf_A.fit_transform(train["achados"].values)
X_test_A  = tfidf_A.transform(test["achados"].values)

# Full text TF-IDF
tfidf_F = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
X_train_F = tfidf_F.fit_transform(train["full"].values)
X_test_F  = tfidf_F.transform(test["full"].values)

print(f"SVC-A features: {X_train_A.shape[1]}")
print(f"SVC-F features: {X_train_F.shape[1]}")

In [ ]:
# =============================================================================
# Apenas 2 SVCs com GroupKFold CV
# =============================================================================

gkf = GroupKFold(n_splits=N_FOLDS)
folds = list(gkf.split(X_train_A, y, groups))

oof_probas = {}
test_probas = {}

t0 = time.time()

for name, X_tr, X_te in [('svc_A', X_train_A, X_test_A), ('svc_F', X_train_F, X_test_F)]:
    print(f"\n--- {name} ---")
    oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
    te_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
    for fi, (tr_idx, va_idx) in enumerate(folds):
        m = CalibratedClassifierCV(
            LinearSVC(class_weight="balanced", random_state=42, max_iter=10000),
            cv=3, method='sigmoid'
        )
        m.fit(X_tr[tr_idx], y[tr_idx])
        oof[va_idx] = m.predict_proba(X_tr[va_idx])
        te_acc += m.predict_proba(X_te) / N_FOLDS
    oof_f1 = f1_score(y, np.argmax(oof, axis=1), average='macro')
    print(f"  OOF F1: {oof_f1:.4f}")
    oof_probas[name] = oof
    test_probas[name] = te_acc

elapsed = time.time() - t0
print(f"\nTrained in {elapsed:.0f}s")

In [ ]:
# =============================================================================
# Blend simples: 40% achados + 60% full (full text é mais forte)
# =============================================================================

oof_blend = 0.40 * oof_probas['svc_A'] + 0.60 * oof_probas['svc_F']
te_blend  = 0.40 * test_probas['svc_A'] + 0.60 * test_probas['svc_F']

baseline_preds = np.argmax(oof_blend, axis=1)
baseline_f1 = f1_score(y, baseline_preds, average='macro')
print(f"OOF macro-F1 (no thresholds): {baseline_f1:.4f}")
print(classification_report(y, baseline_preds, digits=4))

# Threshold APENAS para classe 6 (a mais impactante)
threshold_range = np.arange(0.05, 0.50, 0.01)
best_f1 = baseline_f1
best_t6 = None

for t6 in threshold_range:
    preds = np.argmax(oof_blend, axis=1).copy()
    preds[oof_blend[:, 6] > t6] = 6
    score = f1_score(y, preds, average='macro')
    if score > best_f1:
        best_f1 = score
        best_t6 = t6

if best_t6:
    print(f"\nClass 6 threshold: {best_t6:.2f} -> F1={best_f1:.4f}")
else:
    print("\nNo threshold improvement for class 6")

# Testar adicionar threshold para classe 5
best_t5 = None
for t5 in threshold_range:
    preds = np.argmax(oof_blend, axis=1).copy()
    if best_t6:
        preds[oof_blend[:, 6] > best_t6] = 6
    mask5 = (oof_blend[:, 5] > t5) & (preds != 6)
    preds[mask5] = 5
    score = f1_score(y, preds, average='macro')
    if score > best_f1:
        best_f1 = score
        best_t5 = t5

if best_t5:
    print(f"Class 5 threshold: {best_t5:.2f} -> F1={best_f1:.4f}")

print(f"\nFinal OOF F1: {best_f1:.4f}")

In [ ]:
# =============================================================================
# Submission
# =============================================================================

test_preds = np.argmax(te_blend, axis=1).copy()
if best_t6:
    test_preds[te_blend[:, 6] > best_t6] = 6
if best_t5:
    mask5 = (te_blend[:, 5] > best_t5) & (test_preds != 6)
    test_preds[mask5] = 5

# Clinical guardrail mínimo
def apply_safe_rules(row):
    text = str(row['report']).lower()
    current_pred = int(row['target'])
    if re.search(r'resultado de cine grau 3|carcinoma|\bcdis\b', text):
        return 6
    return current_pred

test['target'] = test_preds
test['target'] = test.apply(apply_safe_rules, axis=1)

submission = test[['ID', 'target']].copy()
submission['target'] = submission['target'].astype(int)
submission.to_csv('submission.csv', index=False)

print('Prediction distribution:')
print(submission['target'].value_counts().sort_index())
print(f'\nSubmission saved! Shape: {submission.shape}')
print(submission)